# Attention heads of the attn-only 2L model

Looks at the attention patterns of `callummcdougall/attn_only_2L_half` and finds its induction
heads.

- Needs the `viz` extra for circuitsvis: `uv sync --extra viz`.
- The model is loaded by `in_tfm.attn_only`, which reproduces TransformerLens 2.17.0's forward pass
  without taking that dependency. See the module docstring for why.
- Everything here runs on CPU in well under a minute. There is no GPU path to set up.

In [ ]:
import circuitsvis as cv
import torch

from in_tfm.attn_only import attention_patterns, load_attn_only_2l

model, tokenizer = load_attn_only_2l()
n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
print(f"{n_layers} layers x {n_heads} heads, d_model={model.config.hidden_size}, "
      f"head_dim={model.config.head_dim}")

## Is this the model you remember?

Two numbers identify it without needing to look at any weights.

- Loss on ordinary english should sit well below the uniform baseline of `log(50278) = 10.83`.
  Somewhere around 5 is right for a 2-layer attention-only model.
- Loss on the second copy of a repeated random sequence should collapse. The model cannot know the
  random tokens, so the only way to predict them is to copy from the first copy. That gap is
  induction, and it is the whole reason this model is a teaching example.

In [ ]:
def token_losses(input_ids: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        logits = model(input_ids=input_ids).logits
    logprobs = logits[:, :-1].log_softmax(-1)
    return -logprobs.gather(-1, input_ids[:, 1:, None])[0, :, 0]


ENGLISH = (
    "The Pile is a large corpus of english text. Machine learning models are trained on it to "
    "predict the next word in a sentence, which we call induction. The results suggest that "
    "attention heads compose with one another across layers."
)
english_ids = torch.tensor([[tokenizer.bos_token_id] + tokenizer(ENGLISH)["input_ids"]])

SEQ = 40
random_half = torch.randint(1000, 40000, (1, SEQ), generator=torch.Generator().manual_seed(0))
repeated_ids = torch.cat(
    [torch.tensor([[tokenizer.bos_token_id]]), random_half, random_half], dim=1
)

repeated_losses = token_losses(repeated_ids)
print(f"uniform baseline        {torch.tensor(50278.0).log():.2f}")
print(f"english                 {token_losses(english_ids).mean():.2f}")
print(f"repeated, first copy    {repeated_losses[1:SEQ + 1].mean():.2f}")
print(f"repeated, second copy   {repeated_losses[SEQ + 1:].mean():.2f}")

## Attention patterns on ordinary text

`attention_patterns` rebuilds the probabilities from the hooked `q_proj` and `k_proj` outputs
rather than from the forward pass, which discards them. It is exact rather than approximate, and it
was checked against TransformerLens's `hook_pattern` to `1e-06`.

Hover a token to follow where a head sends its attention.

In [ ]:
def as_strings(input_ids: torch.Tensor) -> list[str]:
    return [tokenizer.decode([i]) for i in input_ids[0].tolist()]


SHORT = "The war ended in 1945. After the war, the city was rebuilt."
short_ids = torch.tensor([[tokenizer.bos_token_id] + tokenizer(SHORT)["input_ids"]])

cv.attention.attention_heads(
    attention=attention_patterns(model, short_ids, layer_idx=0)[0],
    tokens=as_strings(short_ids),
    attention_head_names=[f"0.{h}" for h in range(n_heads)],
)

In [ ]:
cv.attention.attention_heads(
    attention=attention_patterns(model, short_ids, layer_idx=1)[0],
    tokens=as_strings(short_ids),
    attention_head_names=[f"1.{h}" for h in range(n_heads)],
)

## Scoring every head

Three behaviours, measured on the repeated random sequence.

- **Induction**: a query in the second copy attends to the token that *followed* the previous
  occurrence of itself, so from position `p` to position `p - SEQ + 1`. This is the behaviour that
  lets the model copy.
- **Previous token**: a query attends to the token immediately before it. This is the layer-0 half
  of the circuit, the head that writes "what came before me" into the residual stream so a layer-1
  head can key on it.
- **To BOS**: attention parked on position 0. Heads with nothing to do tend to dump probability
  there, so a high value here is mostly a sign of an inactive head rather than a behaviour.

In [ ]:
rows = torch.arange(SEQ + 1, 2 * SEQ + 1)
induction_cols = rows - SEQ + 1
prev_token_cols = rows - 1

scores = []
for layer in range(n_layers):
    patterns = attention_patterns(model, repeated_ids, layer_idx=layer)[0]
    for head in range(n_heads):
        pattern = patterns[head]
        scores.append(
            {
                "head": f"{layer}.{head}",
                "induction": pattern[rows, induction_cols].mean().item(),
                "prev_token": pattern[rows, prev_token_cols].mean().item(),
                "to_bos": pattern[1:, 0].mean().item(),
            }
        )

print(f"{'head':>6}  {'induction':>10}  {'prev_token':>11}  {'to_bos':>8}")
for row in sorted(scores, key=lambda r: -r["induction"]):
    print(f"{row['head']:>6}  {row['induction']:10.4f}  {row['prev_token']:11.4f}  "
          f"{row['to_bos']:8.4f}")

## The circuit

The table separates cleanly, with no ambiguity to argue about.

- **1.10** and **1.4** are the induction heads, at roughly `0.87` and `0.65`. Every other head
  scores under `0.04`.
- **0.7** is a near-perfect previous-token head, at about `0.997`.

That is the textbook two-layer induction circuit: a previous-token head in layer 0 feeding induction
heads in layer 1. The cell below shows those three heads side by side on the repeated sequence, where
the induction heads should show a stripe offset from the diagonal and `0.7` should show a stripe
directly on it.

In [ ]:
circuit = {"0.7": (0, 7), "1.4": (1, 4), "1.10": (1, 10)}
by_layer = {
    layer: attention_patterns(model, repeated_ids, layer_idx=layer)[0]
    for layer in range(n_layers)
}
selected = torch.stack([by_layer[layer][head] for layer, head in circuit.values()])

cv.attention.attention_heads(
    attention=selected,
    tokens=as_strings(repeated_ids),
    attention_head_names=list(circuit),
)

## Notes

- Head indices here are `layer.head`, and a `q_proj` neuron index in the report pipeline is
  `head * head_dim + dim`. So head `1.10` owns neurons `640` to `703` of layer 1.
- This model has no `q_norm` and no RoPE, and its positional embeddings are shortformer, meaning
  position enters only the `q_proj` and `k_proj` inputs. A query coordinate's activation therefore
  splits exactly into a token term and a position term.